#### Download datasets
##### download SberSQuAD

In [1]:
import pandas as pd
df1 = pd.read_json('./raw_datasets/sberquad/sberquad_train.json')
df2 = pd.read_json('./raw_datasets/sberquad/sberquad_test.json')
df3 = pd.read_json('./raw_datasets/sberquad/sberquad_valid.json')

sbersquad_df = pd.concat([df1, df2, df3], ignore_index=True)   
sbersquad_df['answer'] = sbersquad_df["answers"].apply(lambda x: x["text"][0])
sbersquad_df = sbersquad_df[["context", "question", "answer"]].copy()

no_answ_df = sbersquad_df[sbersquad_df["answer"]==''].copy()
no_answ_df = no_answ_df.drop_duplicates(subset="context", keep="first").copy()

sbersquad_df = sbersquad_df[sbersquad_df["answer"]!=''].copy()
sbersquad_df

,context,question,answer
0,В протерозойских отложениях органические остат...,чем представлены органические остатки?,известковыми выделениями сине-зелёных водорослей
1,В протерозойских отложениях органические остат...,что найдено в кремнистых сланцах железорудной ...,"нитевидные водоросли, грибные нити"
2,В протерозойских отложениях органические остат...,что встречается в протерозойских отложениях?,органические остатки
3,В протерозойских отложениях органические остат...,что относится к числу древнейших растительных ...,скопления графито-углистого вещества
4,В протерозойских отложениях органические остат...,как образовалось графито-углистое вещество?,в результате разложения Corycium enigmaticum
...,...,...,...
74295,Классическая трёхуровневая система накачки раб...,Где используется классическая трёхуровневая си...,в рубиновом лазере.
74296,Первая платёжная карта American Express появил...,какой инвестиционный банк входил в состав Amer...,Lehman Brothers
74297,Следующий альбом Heroes был во многом созвучен...,С каким альбомом был созвучен альбом Дэвида Бо...,Low
74298,"Одним из тех, на кого игра Данна произвела неи...",В каком техасском оркестре выступал гитарист Л...,Light Crust Doughboys


In [2]:
no_answ_df

,context,question,answer
45328,Многоклеточный организм — внесистематическая к...,У каких организмов отсутствуют настоящие диффе...,
45341,Акционерное общество открытого типа АКБ „Моско...,Когда была получена лицензия на осуществление ...,
45346,Однако не лишён этот язык и недостатков: так у...,"В какой среде рожденный, он стимулирует соотве...",
45355,Даже простая убеждённость в чём-либо на основе...,На основе чего простая убеждённость в чём-либо...,
45360,"Транспорт — густая сеть железных дорог, морско...",Какие украшения характерны для конька крыши фл...,
...,...,...,...
69239,В СССР серийно выпускались тепловозы ТЭ1 (1000...,Какая мощность была у серийно выпущенных тепло...,
69244,Мясо собаки являлось традиционным пищевым прод...,"Куда в Азии, согласно кулинарным традициям, по...",
69249,При написании Германии и этнографическо-геогра...,Каким является вопрос о роли личного опыта в о...,
69254,Банк ВТБ является головной структурой Группы В...,Где находятся дочерние и ассоциированные банки...,


##### Download WikiOmnia

In [3]:
from datasets import load_dataset

wo1_df = load_dataset(
    "json",
    data_files={
        "train": "https://huggingface.co/datasets/RussianNLP/wikiomnia/resolve/main/dummy/wikiomnia_ruT5_filtered/wikiomnia_ruT5_filtered_train.json"
    }
)["train"].to_pandas()

wo2_df = load_dataset(
    "json",
    data_files={
        "train": "https://huggingface.co/datasets/RussianNLP/wikiomnia/resolve/main/dummy/wikiomnia_ruGPT3_filtered/wikiomnia_ruGPT_3_filtered_train.json"
    }
)["train"].to_pandas()

wo3_df = load_dataset(
    "json",
    data_files={
        "train": "https://huggingface.co/datasets/RussianNLP/wikiomnia/resolve/main/dummy/wikiomnia_ruT5_raw/wikiomnia_dev.json"
    }
)["train"].to_pandas()

wo4_df = load_dataset(
    "json",
    data_files={
        "train": "https://huggingface.co/datasets/RussianNLP/wikiomnia/resolve/main/dummy/wikiomnia_ruT5_raw/wikiomnia_test.json"
    }
)["train"].to_pandas()

#### Prepare main dataset

In [4]:
import re, pandas as pd
pd.reset_option("display.max_colwidth")

def repetition_score(text):
    text = re.sub(r'[^a-zа-яё0-9 ]', ' ', text.lower()).split()
    return len(set(text)) / len(text)

# ---- concat raw datasets ----
wo_df = pd.concat([wo1_df, wo2_df, wo3_df, wo4_df], ignore_index=True)   
wo_df = wo_df[["summary", "question", "answer",]].rename(
    columns={"summary": "context"}
)
full_df = pd.concat([wo_df, sbersquad_df], ignore_index=True)

# ---- drop deduplicates ----
full_df["answer_lower"] = full_df["answer"].str.lower()
full_df = full_df.drop_duplicates(subset="answer_lower", keep="first").copy()
full_df["question_lower"] = full_df["question"].str.lower()
full_df = full_df.drop_duplicates(subset="question_lower", keep="first").copy()
full_df = full_df.drop(columns=["answer_lower", "question_lower"]).copy()

# ---- drop short sentences ----
min_cont_len = 240
full_df = full_df[full_df["answer"].fillna("").str.len() > 30]
full_df = full_df[full_df["context"].str.len() > min_cont_len]
full_df = full_df[(full_df["answer"].str.len() / full_df["context"].str.len()) < 0.8]

# drop short sentences in no answer data
no_answ_df = no_answ_df[no_answ_df["context"].str.len() > min_cont_len]

# ---- drop context with formula like {...} ---- 
pattern = r"[{}]"
mask = full_df['context'].str.contains(pattern, regex=True, na=False)
full_df = full_df[~mask].copy()

# drop formula in no answer data
mask = no_answ_df['context'].str.contains(pattern, regex=True, na=False)
no_answ_df = no_answ_df[~mask].copy()

# ---- drop context with list of source ----
pattern = r"//[^//]+//"
mask = full_df['context'].str.contains(pattern, regex=True, na=False)
full_df = full_df[~mask].copy()

# drop context with list of source in no answer data
mask = no_answ_df['context'].str.contains(pattern, regex=True, na=False)
no_answ_df = no_answ_df[~mask].copy()

# ---- drop repetition text ---- 
threshold = 0.6
for col in ["context", "question", "answer"]:
    full_df = full_df[full_df[col].map(repetition_score) > threshold]

# drop repetition text in no answer data
no_answ_df = no_answ_df[no_answ_df['context'].map(repetition_score) > threshold]

full_df

,context,question,answer
8,Тиопентал натрия (лат. Thiopentalum-natrium) —...,Что такое тиопентал натрия?,средство для неингаляционного наркоза ультрако...
10,Алгоритмическая разрешимость — свойство формал...,"Что является частным, но вместе с тем важнейши...",Вопрос о выводимости в формальной теории
11,Алгоритмическая разрешимость — свойство формал...,Чем является вопрос о выводимости в формальной...,"частным, но вместе с тем важнейшим случаем бол..."
21,Пушечный двор (Москва) — центр пушечно-литейно...,Где существовали производства пушечно-литейног...,"во многих городах сторон (краях, странах) Русс..."
26,В этом телужском имени фамилия (Пентала) стоит...,Кто такой Пентала Харикришна?,"индийский шахматист, гроссмейстер"
...,...,...,...
2845734,"Литва планировала перейти на евро в 2007 году,...",Из-за чего задержался переход на евро?,Из-за незначительного превышения Маастрихтских...
2845735,В 2002 году газета Philadelphia Inquirer со сс...,Когда предположительно мог быть завербован Фишер?,во время поездки в Москву в 1958 году
2845737,В Вильнюсе открыто несколько десятков старинны...,Какие монастыри действуют в Вильнюсе?,Несколько католических и православный (мужской...
2845740,Хозяйственный расчёт - система экономических о...,Какие важнейшие особенности хозрасчёта?,рентабельность и самоокупаемость


In [5]:
no_answ_df

,context,question,answer
45328,Многоклеточный организм — внесистематическая к...,У каких организмов отсутствуют настоящие диффе...,
45341,Акционерное общество открытого типа АКБ „Моско...,Когда была получена лицензия на осуществление ...,
45346,Однако не лишён этот язык и недостатков: так у...,"В какой среде рожденный, он стимулирует соотве...",
45355,Даже простая убеждённость в чём-либо на основе...,На основе чего простая убеждённость в чём-либо...,
45360,"Транспорт — густая сеть железных дорог, морско...",Какие украшения характерны для конька крыши фл...,
...,...,...,...
69234,На основе прежней конфедерации телеских племён...,Какой года был последним годом существования С...,
69244,Мясо собаки являлось традиционным пищевым прод...,"Куда в Азии, согласно кулинарным традициям, по...",
69249,При написании Германии и этнографическо-геогра...,Каким является вопрос о роли личного опыта в о...,
69254,Банк ВТБ является головной структурой Группы В...,Где находятся дочерние и ассоциированные банки...,


#### Check repetition score and length distribution

In [6]:
print('\nContext repetition score:\n', full_df['context'].map(repetition_score).describe())
print('\nAnswers length:\n', full_df["answer"].str.len().describe())


Context repetition score:
 count    143709.000000
mean          0.836778
std           0.080649
min           0.600386
25%           0.782178
50%           0.840426
75%           0.896552
max           1.000000
Name: context, dtype: float64

Answers length:
 count    143709.000000
mean         48.954679
std          18.089674
min          31.000000
25%          36.000000
50%          43.000000
75%          56.000000
max         258.000000
Name: answer, dtype: float64


##### Define class QACleaner

In [7]:
import re
import unicodedata
import pandas as pd
from qa_instructions import pick_instruction


class QACleaner:
    def __init__(
            self, 
            tokenizer, 
            short_answ_ratio=0.15, 
            long_answ_ratio=0.30, 
            max_answ_len=150,
            max_cont_len=750
        ):

        self.tokenizer = tokenizer
        self.short_answ_ratio = short_answ_ratio
        self.long_answ_ratio = long_answ_ratio
        self.max_answ_len = max_answ_len
        self.max_cont_len = max_cont_len

        # --- replacements ---
        self.replacements = {
            "«": '"',
            "»": '"',
            "“": '"',
            "”": '"',
            "„": '"',
            "’": "'",
        }

        # Remove html tags
        self.remove_tags = re.compile(r'<(?!table>)[^>]+>')  # del all teg except <table>, # (r"<[^>]+>") - dell all teg

        # Regex: remove http limks
        self.remove_http = re.compile(r'(?:https?://|ftp://|www\.)\S*')

        # Regex: remove citation markers like [1] 
        self.remove_citation = re.compile(r"\[[^\[\]]*\]")   

        # Regex: remove empty quotation '', "", " "
        self.remove_empty_quotation = re.compile(r"['\"]\s*['\"]")

        # Regex: remove triple parentheses ( ( ( ) ) ) 
        self.remove_triple_parentheses = re.compile(r"\([^()]*\([^()]*\([^()]*\)[^()]*\)[^()]*\)")

        # Regex: remove double parentheses ( ( ) )
        self.remove_double_parentheses = re.compile(r"\([^()]*\([^()]*\)[^()]*\)")

        self.remove_parentheses = re.compile(
            r'\('
            r'(?!'                                                            # (?!...) — negative lookahead
                r'\s*\d{4}\s*\)'                                              # allow years like (2015), ( 2015 )
                r'|'
                r'\s*[а-яА-ЯёЁ 0-9,.:"\-]*[а-яА-ЯёЁ][а-яА-ЯёЁ 0-9,.:"\-]*\)'  # Cyrillic present, allowed chars only
            r')'
            r'[^)]*'                                                          # get full content in ()
            r'\)'
        )

        # Regex: remove bad marks like == Text ==
        self.remove_bad_marks = re.compile(r"[=]+\s*[=]*[A-Za-zА-ЯЁа-яё]+\s*[=]+\s*[=]*")
         
        # Regex: remove Unicode characters
        self.unicode_chars = re.compile(
            r'[\x00-\x08\x0B-\x0C\x0E-\x1F\x7F-\x9F'
            r'\u200B-\u200F\u202A-\u202E\u2060-\u206F'
            r'\uFE00-\uFE0F\uFEFF\uFFF9-\uFFFB'
            r'\uE000-\uF8FF\uFDD0-\uFDEF]'
        )    

        # Regex: collapse repeated punctuation symbols
        self.double_symbols = re.compile(r'([.,:;/_!?~$@%^&*#+=\-—])(?:\s*\1)+')

        # Regex: fix punctuation chain
        self.fix_punct_chain = re.compile(r'[:;!?,.]{2,}(?=\s*[А-ЯЁA-Z])')

        # Regex: remove rest punctuation chain
        self.remove_punct_chain = re.compile(r'[:;!?,.]{2,}')

        # Regex: separate joined words and punctuation
        self.fix_join = re.compile(
            r"(?<=[а-яё])(?=[А-ЯЁ0-9A-Z])"
            r"|(?<=[0-9])(?=[а-яёА-ЯЁA-Za-z])"
            r"|(?<=[A-Za-z])(?=[А-ЯЁа-яё])"
            r"|(?<=[А-ЯЁа-яё])(?=[A-Za-z])"
            r"|(?<=[^\W\d_][,.!?:;'\")])(?=[^\W\d_])"
        )    

        # Regex: collapse multiple spaces but preserve newline
        self.multi_space = re.compile(r"[^\S\n]+")

        # Regex: remove space before punctuation: " ," -> ","
        self.space_before_punct = re.compile(r"\s+([,.;:!?])")

        # Regex: separate text to sentences
        self.sent_split =  re.compile(r'(?<!\b[А-ЯA-Z])(?<!\b[а-яa-z])(?<!\d)[.!?]+(?=\s+[А-ЯA-Z])') # re.compile(r'[.!?]+')

    # -----------------------------------------------------

    def remove_accents(self, text: str) -> str:
        text = unicodedata.normalize("NFD", text)
        result = []
        for ch in text:
            # remove ONLY acute accent (stress mark)
            if ch == '\u0301':  # COMBINING ACUTE ACCENT
                continue
            result.append(ch)

        return unicodedata.normalize("NFC", "".join(result))

    # -----------------------------------------------------

    def normalize_text(self, text: str) -> str:

        for k, v in self.replacements.items():
            text = text.replace(k, v)

        text = self.remove_citation.sub("", text)

        text = self.remove_triple_parentheses.sub("", text)
        text = self.remove_double_parentheses.sub("", text)
        text = self.remove_parentheses.sub("", text)

        text = self.remove_tags.sub(" ", text)
        text = self.remove_http.sub("", text) 
        text = self.remove_bad_marks.sub("", text) 

        text = text.replace("\xa0", " ")
        text = text.replace("\xad", " ")
        text = self.unicode_chars.sub(" ", text)

        text = self.double_symbols.sub(r"\1", text)
        text = self.fix_punct_chain.sub(r".", text)
        text = self.remove_punct_chain.sub(r" ", text)

        text = self.multi_space.sub(" ", text)
        text = self.space_before_punct.sub(r"\1", text)
        text = self.fix_join.sub(" ", text)

        return text.strip()

    # -----------------------------------------------------

    def clean_df(self, df: pd.DataFrame) -> pd.DataFrame:

        # ---- remove accents and normalize ----
        for col in ["context", "question", "answer"]:
            df[col] = df[col].map(self.remove_accents)
            df[col] = df[col].map(self.normalize_text)

        # ---- length filters ----
        df = df[
            (df["answer"].str.len() > 30) &
            (df["context"].str.len() > 240)
        ].copy()

        # ---- preparing instructions ----
        df["answer_len"] = df["answer"].str.len()
        df["context_tokens"] = df["context"].map(self._encoded_text_len)

        # separation to short and long answers
        short_answers = df[
            (df["answer_len"] < 45) & (df["context_tokens"] <= self.max_answ_len)
        ]
        long_answers  = df[
            (df["answer_len"] >= 45) & (df["context_tokens"] <= self.max_answ_len)
        ]

        # Sample short and long indices
        short_n = int(self.short_answ_ratio * df.shape[0])
        long_n  = int(self.long_answ_ratio * df.shape[0])
        short_idx = short_answers.sample(n=min(short_n, len(short_answers)), random_state=42).index
        long_idx = long_answers.sample(n=min(long_n, len(long_answers)), random_state=42).index

        # replace answer for long answers
        df.loc[long_idx, "answer"] = df.loc[long_idx, "context"]

        # assign instruction types
        df["instruction_type"] = "default"
        df.loc[short_idx, "instruction_type"] = "short"
        df.loc[long_idx,  "instruction_type"] = "detail"
        
        # map instruction
        df["instruction"] = df["instruction_type"].apply(pick_instruction)

        # ---- align answers to content for default instructions ----
        mask_default = df["instruction_type"] == "default"
        df.loc[mask_default, "answer"] = [
            self.align_answer_to_context(c, a)
            for c, a in zip(
                df.loc[mask_default, "context"],
                df.loc[mask_default, "answer"]
            )
        ]

        # ---- answer must be in context ----
        df = df[df.apply(lambda x: x["answer"].lower() in x["context"].lower(), axis=1)].copy()

        # ---- drop too long answers ----
        df = df[df['answer'].map(self._encoded_text_len) <= self.max_answ_len].copy()

        # ---- drop too long context ----
        df = df[df['context_tokens'] <= self.max_cont_len].copy()

        return df.reset_index(drop=True)
    
    # -----------------------------------------------------
    
    def _uniq_words(self, text: str) -> set[str]:
        text = re.sub(r'[^a-zа-яё0-9 ]', ' ', text.lower())
        text = re.sub(r'\s+', ' ', text).strip()
        return set(text.split())
    
    # -----------------------------------------------------

    def _overlap_score(self, answer_words: set, sent_words: set) -> float:
        if not answer_words:
            return 0.0
        return len(answer_words & sent_words) / len(answer_words)
    
    # -----------------------------------------------------

    def _encoded_text_len(self, text):
        return len(self.tokenizer.encode(text))

    # -----------------------------------------------------    

    def align_answer_to_context(self, context: str, answer: str, threshold: float = 0.8) -> str:

        answer_words = self._uniq_words(answer)
        sentences = [s.strip() for s in self.sent_split.split(context) if s.strip()]
        best_sent = answer
        best_score = 0.0

        for sent in sentences:
            sent_words = self._uniq_words(sent)
            score = self._overlap_score(answer_words, sent_words)

            if score > best_score:
                best_score = score
                best_sent = sent

        return best_sent if best_score >= threshold else answer

#### Clean main dataset

In [8]:
pd.set_option("display.max_colwidth", None)
from bpe_tokenizer import BPETokenizer

bpe_tokenizer = BPETokenizer().from_file("d:/Neuro_net/Projects/LLM/Gen/tokenizer/bpe_tokenizer_v3/tokenizer.json")
bpe_tokenizer.post_processor = None

cleaner = QACleaner(bpe_tokenizer)

full_df = cleaner.clean_df(full_df)
full_df

,context,question,answer,answer_len,context_tokens,instruction_type,instruction
0,"Тиопентал натрия — средство для неингаляционного наркоза ультракороткого действия. Представляет собой смесь натриевой соли -5-(1-метилбутил)-5-этил-2-тиобарбитуровой кислоты с безводным натрия карбонатом. Сверхтерапевтические (летальные) дозы широко используются для усыпления животных, в США является одним из трёх компонентов смертельной инъекции (смертная казнь) вместе с панкуронием и хлоридом калия.",Что такое тиопентал натрия?,"Тиопентал натрия — средство для неингаляционного наркоза ультракороткого действия. Представляет собой смесь натриевой соли -5-(1-метилбутил)-5-этил-2-тиобарбитуровой кислоты с безводным натрия карбонатом. Сверхтерапевтические (летальные) дозы широко используются для усыпления животных, в США является одним из трёх компонентов смертельной инъекции (смертная казнь) вместе с панкуронием и хлоридом калия.",62,116,detail,Ответь подробно только из данных документов.
1,"Алгоритмическая разрешимость — свойство формальной теории обладать алгоритмом, определяющим по данной формуле, выводима она из множества аксиом данной теории или нет. Теория называется разрешимой, если такой алгоритм существует, и неразрешимой, в противном случае. Вопрос о выводимости в формальной теории является частным, но вместе с тем важнейшим случаем более общей проблемы разрешимости.","Что является частным, но вместе с тем важнейшим случаем более общей проблемы разрешимости?",Вопрос о выводимости в формальной теории,40,89,short,"Кратко ответь из контекста, если ответа нет, скажи об этом."
2,"Алгоритмическая разрешимость — свойство формальной теории обладать алгоритмом, определяющим по данной формуле, выводима она из множества аксиом данной теории или нет. Теория называется разрешимой, если такой алгоритм существует, и неразрешимой, в противном случае. Вопрос о выводимости в формальной теории является частным, но вместе с тем важнейшим случаем более общей проблемы разрешимости.",Чем является вопрос о выводимости в формальной теории?,"Алгоритмическая разрешимость — свойство формальной теории обладать алгоритмом, определяющим по данной формуле, выводима она из множества аксиом данной теории или нет. Теория называется разрешимой, если такой алгоритм существует, и неразрешимой, в противном случае. Вопрос о выводимости в формальной теории является частным, но вместе с тем важнейшим случаем более общей проблемы разрешимости.",76,89,detail,Дай детальный ответ из данных документов.
3,"Пушечный двор (Москва) — центр пушечно-литейного и колокольного производства в России в XVI—XVII веках. Под данным словосочетанием существовали производства (центры) литейного дела во многих городах сторон (краях, странах) Русского государства (на Псковщине, Новгородчине, Вологотчине и так далее).",Где существовали производства пушечно-литейного и колокольного производства?,"Пушечный двор (Москва) — центр пушечно-литейного и колокольного производства в России в XVI—XVII веках. Под данным словосочетанием существовали производства (центры) литейного дела во многих городах сторон (краях, странах) Русского государства (на Псковщине, Новгородчине, Вологотчине и так далее).",62,70,detail,Дай полный ответ из данного контекста.
4,"В этом телужском имени фамилия (Пентала) стоит перед личным именем. Пентала Харикришна — индийский шахматист, гроссмейстер (2001). Чемпион мира среди мальчиков до 10 лет (1996). Чемпион Азии среди мальчиков до 14 лет (2000). Чемпион Содружества (2001). Чемпион мира среди юношей (2004). Чемпион Индии (2004). В составе национальной сборной участник 6-и Олимпиад. Победитель Pokerstars International Open 2015.",Кто такой Пентала Харикришна?,"индийский шахматист, гроссмейстер",33,95,short,"Дай короткий ответ, используя эти документы."
...,...,...,...,...,...,...,...
123312,"Литва планировала перейти на евро в 2007 году, но из-за незначительного превышения Маастрихтских критериев по инфляции переход пришлось отложить, и она ввела евро только 1 января 2015 года. Латвия с

#### Clean no answers dataset 

In [9]:
from qa_instructions import pick_instruction

# ---- remove accents and normalize ----
no_answ_df['context'] = no_answ_df['context'].map(cleaner.remove_accents)
no_answ_df['context'] = no_answ_df['context'].map(cleaner.normalize_text)

# ---- length filter ----
no_answ_df["context_tokens"] = no_answ_df["context"].map(cleaner._encoded_text_len)
no_answ_df = no_answ_df[
    (no_answ_df["context_tokens"] > 90) & (no_answ_df["context_tokens"] < 200)
].copy()

# sample up to 2% no answers
n = int(0.02 * full_df.shape[0])
no_answ_df = no_answ_df.sample(n=min(n, len(no_answ_df)), random_state=42).copy()

# map instruction
no_answ_df["instruction_type"] = "not_found"
no_answ_df["instruction"] = no_answ_df["instruction_type"].apply(pick_instruction)

# drop unnecessary columns
no_answ_df.drop(columns=['context_tokens', 'instruction_type'], inplace=True)

no_answ_df

,context,question,answer,instruction
66984,"Первые версии UNIX были написаны на ассемблере и не имели встроенного компилятора с языком высокого уровня. Примерно в 1969 году Кен Томпсон при содействии Денниса Ритчи разработал и реализовал язык Би, представлявший собой упрощённый (для реализации на мини-компьютерах) вариант разработанного в 1966 языка BCPL. Би, как и BCPL, был интерпретируемым языком. В 1972 году была выпущена вторая редакция UNIX, переписанная на языке Би. В 1969—1973 гг. на основе Би был разработан компилируемый язык, получивший название Си.","Чем был Би, как и BCPL?",,"Ответь кратко, если ответа нет в контексте, так и скажи."
47740,"Впрочем, попытки введения критериев для выделения цивилизаций предпринимались неоднократно. Российский историк Э. Д. Фролов в одной из своих работ перечислил их наиболее распространённый набор: общность геополитических условий, исконное языковое родство, единство или близость экономического и политического строя, культуры (включая религию) и менталитета. Вслед за Шпенглером и Тойнби учёный признавал, что оригинальное качество цивилизации обусловлено оригинальным свойством каждого из структурообразующих элементов и их неповторимым единством.",Чем обусловлено качество цивилизации по мнению Фролова?,,"Дай полный ответ, если в документах нет ответа, так и скажи."
60757,"Обучение без учителя — позволяет распознать образы во входном потоке. Обучение с учителем включает также классификацию и регрессионный анализ. Классификация используется, чтобы определить, к какой категории принадлежит образ. Регрессионный анализ используется, чтобы в рядах числовых примеров входа/выхода и обнаружить непрерывную функцию, на основании которой можно было бы прогнозировать выход. При обучении агент вознаграждается за хорошие ответы и наказывается за плохие. Они могут быть проанализированы с точки зрения теории решений, используя такие понятия как полезность. Математический анализ машинных алгоритмов изучения — это раздел теоретической информатики, известный как вычислительная теория обучения.",Что содержит обучение с учителем,,"Ответь кратко, только из данных документов."
64497,"Древнегреческая философия оказала определяющее влияние на всю историю западной и отчасти даже мировой философии вплоть до сегодняшнего дня. Самим термином философия мы обязаны именно античности. Расцвет древнегреческой философии приходится на V—IV вв. до н. э а отголоски её замирали ещё в течение тысячелетия. В Византии и странах ислама господствующее влияние греческой философии сохранялось в течение всего следующего тысячелетия; затем, во времена Ренессанса и гуманизма, и в Европе произошло возрождение греческой философии, что привело к творческим новообразованиям, начиная от платонизма и аристотелизма эпохи Ренессанса и кончая влиянием греческой философии на всё развитие европейской философской мысли (см. Европейская философия).",В какие времена в Европе произошло возрождение греческой философии?,,"Ответь детально из этих документов, если ответа нет, скажи."
48541,"После высадки на о. Каяк, пакетбот повернул обратно к берегам Камчатки, следуя вдоль южного берега Аляски и Алеутской гряды. По пути были открыты остров Кадьяк, Евдокеевские и Шумагинские острова, острова Св. Иоанна (Атха), Св. Маркиана (Кыска) и Св. Стефана (Булдырь). 5 ноября пакетбот зашел для пополнения запасов воды на остров, впоследствии названный островом Беринга, где 28 ноября сильным ветром был выброшен на берег. В тяжелых условиях вынужденной зимовки от цинги умерли 19 человек, а 8 декабря скончался и Витус Беринг. Командование принял штурман поручик Свен Ваксель. Весной 1742 года 46 оставшихся (из 75) членов экипажа сумели построить из обломков пакетбота гукор (также названный Св. Петром ) и в августе 1742 года, преодолев 250 км, достигли Авачинской губы.",Кто стал командиром экипажа после смерти Витуса Беринга,,Ответь подробно только из данных документов.
...,...,...,...,...
45866,"Диоген Лаэртский приводит несколько заглавий сочинения Гераклита: Музы, О п

#### Check instruction_type and tokens length distribution

In [10]:
print('Numbers of short and detail (TO DO):', (int(cleaner.short_answ_ratio * full_df.shape[0]), int(cleaner.long_answ_ratio * full_df.shape[0])))
counts = full_df["instruction_type"].value_counts()
print(counts, '\ntotal =', counts.sum()) 
print('\nContext tokens length:\n', full_df['context_tokens'].describe())
print('\nAnswers length:\n', full_df['answer'].map(cleaner._encoded_text_len).describe())

Numbers of short and detail (TO DO): (18497, 36995)
instruction_type
default    69032
detail     36717
short      17568
Name: count, dtype: int64 
total = 123317

Context tokens length:
 count    123317.000000
mean        160.148787
std         108.402489
min          30.000000
25%          85.000000
50%         127.000000
75%         197.000000
max         750.000000
Name: context_tokens, dtype: float64

Answers length:
 count    123317.000000
mean         48.865809
std          38.361592
min           3.000000
25%          18.000000
50%          36.000000
75%          72.000000
max         150.000000
Name: answer, dtype: float64


#### Concat to final dataset

In [11]:
# keep only necessary columns
full_df = full_df[['context', 'question', 'answer', 'instruction']].copy()
full_df = pd.concat([full_df, no_answ_df], ignore_index=True)  
full_df

,context,question,answer,instruction
0,"Тиопентал натрия — средство для неингаляционного наркоза ультракороткого действия. Представляет собой смесь натриевой соли -5-(1-метилбутил)-5-этил-2-тиобарбитуровой кислоты с безводным натрия карбонатом. Сверхтерапевтические (летальные) дозы широко используются для усыпления животных, в США является одним из трёх компонентов смертельной инъекции (смертная казнь) вместе с панкуронием и хлоридом калия.",Что такое тиопентал натрия?,"Тиопентал натрия — средство для неингаляционного наркоза ультракороткого действия. Представляет собой смесь натриевой соли -5-(1-метилбутил)-5-этил-2-тиобарбитуровой кислоты с безводным натрия карбонатом. Сверхтерапевтические (летальные) дозы широко используются для усыпления животных, в США является одним из трёх компонентов смертельной инъекции (смертная казнь) вместе с панкуронием и хлоридом калия.",Ответь подробно только из данных документов.
1,"Алгоритмическая разрешимость — свойство формальной теории обладать алгоритмом, определяющим по данной формуле, выводима она из множества аксиом данной теории или нет. Теория называется разрешимой, если такой алгоритм существует, и неразрешимой, в противном случае. Вопрос о выводимости в формальной теории является частным, но вместе с тем важнейшим случаем более общей проблемы разрешимости.","Что является частным, но вместе с тем важнейшим случаем более общей проблемы разрешимости?",Вопрос о выводимости в формальной теории,"Кратко ответь из контекста, если ответа нет, скажи об этом."
2,"Алгоритмическая разрешимость — свойство формальной теории обладать алгоритмом, определяющим по данной формуле, выводима она из множества аксиом данной теории или нет. Теория называется разрешимой, если такой алгоритм существует, и неразрешимой, в противном случае. Вопрос о выводимости в формальной теории является частным, но вместе с тем важнейшим случаем более общей проблемы разрешимости.",Чем является вопрос о выводимости в формальной теории?,"Алгоритмическая разрешимость — свойство формальной теории обладать алгоритмом, определяющим по данной формуле, выводима она из множества аксиом данной теории или нет. Теория называется разрешимой, если такой алгоритм существует, и неразрешимой, в противном случае. Вопрос о выводимости в формальной теории является частным, но вместе с тем важнейшим случаем более общей проблемы разрешимости.",Дай детальный ответ из данных документов.
3,"Пушечный двор (Москва) — центр пушечно-литейного и колокольного производства в России в XVI—XVII веках. Под данным словосочетанием существовали производства (центры) литейного дела во многих городах сторон (краях, странах) Русского государства (на Псковщине, Новгородчине, Вологотчине и так далее).",Где существовали производства пушечно-литейного и колокольного производства?,"Пушечный двор (Москва) — центр пушечно-литейного и колокольного производства в России в XVI—XVII веках. Под данным словосочетанием существовали производства (центры) литейного дела во многих городах сторон (краях, странах) Русского государства (на Псковщине, Новгородчине, Вологотчине и так далее).",Дай полный ответ из данного контекста.
4,"В этом телужском имени фамилия (Пентала) стоит перед личным именем. Пентала Харикришна — индийский шахматист, гроссмейстер (2001). Чемпион мира среди мальчиков до 10 лет (1996). Чемпион Азии среди мальчиков до 14 лет (2000). Чемпион Содружества (2001). Чемпион мира среди юношей (2004). Чемпион Индии (2004). В составе национальной сборной участник 6-и Олимпиад. Победитель Pokerstars International Open 2015.",Кто такой Пентала Харикришна?,"индийский шахматист, гроссмейстер","Дай короткий ответ, используя эти документы."
...,...,...,...,...
125778,"Диоген Лаэртский приводит несколько заглавий сочинения Гераклита: Музы, О природе, Правило негрешимое уставу жить и ещё ряд вариантов; скорее всего, все они не принадлежат автору. Он же пишет о том, что поэма Гераклита разделяется на три рассуждения: обо всём, о государстве и о божестве. По его словам, Гераклит поме

In [431]:
import torch
import numpy as np
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel

@torch.no_grad()
def embed_texts(texts):
    vectors = []
    tokenizer = AutoTokenizer.from_pretrained("intfloat/multilingual-e5-large")
    model = AutoModel.from_pretrained("intfloat/multilingual-e5-large") 
    model.eval()
    def mean_pooling(model_output, attention_mask):
        token_embeddings = model_output.last_hidden_state
        mask = attention_mask.unsqueeze(-1).float()
        summed = (token_embeddings * mask).sum(dim=1)
        counts = mask.sum(dim=1).clamp(min=1e-9)
        return summed / counts
    
    for _, text in texts:
        batch = tokenizer(
            text,
            max_length=512,
            truncation=True,
            padding=True,
            return_tensors="pt",
        )
        outputs = model(**batch)
        embeddings = mean_pooling(outputs, batch["attention_mask"])
        embeddings = F.normalize(embeddings, p=2, dim=1) # L2 norm
        vectors.append(embeddings.cpu().numpy().astype(np.float32))

    return np.vstack(vectors)

In [583]:
def tokenize(text: str) -> set[str]:
        text = text.lower()
        text = re.sub(r'[^a-zа-яё0-9 ]', ' ', text)
        text = re.sub(r'\s+', ' ', text).strip()
        return set(text.split())

def overlap_score(answer_words: set, sent_words: set) -> float:
        return len(answer_words & sent_words) / len(answer_words)

SENT_SPLIT = re.compile(r'(?<!\b[А-ЯA-Z])(?<!\b[а-яa-z])(?<!\d)[.!?]+(?=\s+[А-ЯA-Z])')
i = 121827
question = full_df['question'][i]
context = full_df['context'][i]
answer = tokenize(full_df['answer'][i])
sentences = [s.strip() for s in SENT_SPLIT.split(context)]
best_sent = answer
best_score = 0.0
result = []

print(question, '\n', full_df['answer'][i], "\n", answer, "\n")
for sent in sentences:
    if sent:
        sent_words  = tokenize(sent)
        score = overlap_score(answer, sent_words)
        result.append([score, sent])

# result.sort(key=lambda x: x[0], reverse=True)  # sort by score descending

for score, sent in result:
    print(f"{score:.4f} | {sent}")

Над каким словарём работали Ю. Герулис и Э. Френкель? 
 Исследованиями балтийских языков и их связей со славянскими и другими индоевропейскими языками занимались Р. Траутман ( Балто-славянский словарь ), Ю. Герулис, Э. Френкель ( Литовский этимологический словарь ), К. Станг (первая Сравнительная грамматика балтийских языков ), Х. Педерсен, Т. Торбьёрнссон, М. Фасмер, Э. Герман, Э. Ниеминец, Е. Курилович, Я. Отрембский, П. Арумаа, В. Кипарский, А. Зенн, Ю. Бальчиконис, П. Скаржюс, А. Салис, П. Йоникас, Ю. Плакис, Э. Блесе, А. Аугсткалнис, А. Абеле, В. Руке-Дравиня, К. Дравиньш, В. Мажулис, З. Зинкявичюс, Й. Казлаускас, Вяч 
 {'зинкявичюс', 'языков', 'педерсен', 'к', 'исследованиями', 'герман', 'связей', 'руке', 'ю', 'казлаускас', 'аугсткалнис', 'балто', 'й', 'славянскими', 'сравнительная', 'славянский', 'скаржюс', 'дравиньш', 'в', 'х', 'блесе', 'ниеминец', 'я', 'торбьёрнссон', 'плакис', 'вяч', 'м', 'мажулис', 'их', 'литовский', 'отрембский', 'е', 'фасмер', 'герулис', 'п', 'словарь', 'э

##### checking the proportion of no_answer

In [12]:
no_answer_mask = full_df["answer"].astype(str).str.strip() == ""
df_no = full_df[no_answer_mask]
df_yes = full_df[~no_answer_mask]
print(len(df_yes), len(df_no), len(df_no) / len(full_df))

123317 2466 0.019605193070605726


#### Save final dataset to file

In [13]:
full_df.to_parquet("full_df.parquet", index=False)

#### Read dataset from file

In [1]:
import pandas as pd
pd.reset_option("display.max_colwidth")

full_df = pd.read_parquet("full_df.parquet")
full_df

,context,question,answer,instruction
0,Тиопентал натрия — средство для неингаляционно...,Что такое тиопентал натрия?,Тиопентал натрия — средство для неингаляционно...,Ответь подробно только из данных документов.
1,Алгоритмическая разрешимость — свойство формал...,"Что является частным, но вместе с тем важнейши...",Вопрос о выводимости в формальной теории,"Кратко ответь из контекста, если ответа нет, с..."
2,Алгоритмическая разрешимость — свойство формал...,Чем является вопрос о выводимости в формальной...,Алгоритмическая разрешимость — свойство формал...,Дай детальный ответ из данных документов.
3,Пушечный двор (Москва) — центр пушечно-литейно...,Где существовали производства пушечно-литейног...,Пушечный двор (Москва) — центр пушечно-литейно...,Дай полный ответ из данного контекста.
4,В этом телужском имени фамилия (Пентала) стоит...,Кто такой Пентала Харикришна?,"индийский шахматист, гроссмейстер","Дай короткий ответ, используя эти документы."
...,...,...,...,...
125778,Диоген Лаэртский приводит несколько заглавий с...,"На сколько суждений, по словам Диогена Лаэртск...",,Ответь используя только данные документы.
125779,Поскольку на реальном рынке трудно выявить ист...,Какой термин используется при обнаружении паде...,,"Кратко ответь из контекста, если ответа нет, с..."
125780,"В то же время, лидеры отрасли АВТОВАЗ, ГАЗ и А...",Какая часть российских автомобильных и моторны...,,Ответь подробно только из данных документов.
125781,Фест. По значению для археологической науки Фе...,Каким был Фест?,,Ответь используя только данные документы.


#### Build QA examples

In [ ]:
# from __future__ import annotations
import faiss
from tqdm import tqdm
from typing import List, Dict, Any, Optional
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel


class QASampleBuilder:
    def __init__(
        self,
        df: pd.DataFrame,
        model_name: str = "intfloat/multilingual-e5-large",
        top_k: int = 8,
        batch_size: int = 8,
        max_length: int = 512,
        build_new_index: bool = True,
        q_emb_path: str = "question_embeddings.npy",
        no_answer_text: str = "К сожалению, ответ не найден.",
        device: Optional[str] = None,
    ) -> None:
        self.df = df.copy()
        self.top_k = top_k
        self.batch_size = batch_size
        self.max_length = max_length
        self.build_new_index = build_new_index
        self.q_emb_path = q_emb_path
        self.no_answer_text = no_answer_text
        self.device = torch.device(
            device if device is not None else ("cuda" if torch.cuda.is_available() else "cpu")
        )
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModel.from_pretrained(model_name).to(self.device)
        self.model.eval()
        self.unique_contexts: Optional[pd.DataFrame] = None
        self.context_embeddings: Optional[np.ndarray] = None
        # In-memory mapping from context text -> doc_id
        self.context_to_doc_id: Optional[dict[str, int]] = None
        

    def build(self) -> List[Dict[str, Any]]:
        self._build_context_embeddings()
        self._build_faiss_index()          
        return self._build_qa_examples()

    def _build_context_embeddings(self) -> None:
        unique_context_list = self.df["context"].drop_duplicates().tolist()

        self.unique_contexts = pd.DataFrame({
            "doc_id": np.arange(len(unique_context_list), dtype=np.int64),
            "context": unique_context_list,
        })

        self.context_to_doc_id = dict(
            zip(self.unique_contexts["context"], self.unique_contexts["doc_id"])
        )
        
        if self.build_new_index:
            self.context_embeddings = self._embed_contexts(unique_context_list)
        else:
            self.context_embeddings = None

    def _build_faiss_index(self) -> None:
        if self.build_new_index:
            if self.context_embeddings is None:
                raise RuntimeError("Context embeddings not built")
            
            # cosine similarity = inner product (because vectors are normalized)
            self.index = faiss.IndexFlatIP(self.context_embeddings.shape[1])
            self.index.add(self.context_embeddings)
            faiss.write_index(self.index, "index.faiss")
        else:
            self.index = faiss.read_index("index.faiss")

    def _build_qa_examples(self) -> List[Dict[str, Any]]:
        if self.unique_contexts is None or self.context_to_doc_id is None:
            raise RuntimeError("Unique_contexts must be built first")

        if self.build_new_index:
            questions = self.df["question"].tolist()
            question_embeddings = self._embed_queries(questions)
            # save compressed
            np.save(self.q_emb_path, question_embeddings.astype(np.float16))
        else:
            question_embeddings = np.load(self.q_emb_path).astype(np.float32)

        results: List[Dict[str, Any]] = []
        candidate_k = self.top_k * 3  # larger buffer for filtering
        batch_size = 1000

        for start in tqdm(range(0, len(self.df), batch_size), desc='Build_qa_examples'):
            end = min(start + batch_size, len(self.df))
            q_batch = question_embeddings[start:end]

            # FAISS search
            scores, indices = self.index.search(q_batch, candidate_k)

            for i, row_idx in enumerate(range(start, end)):
                row = self.df.iloc[row_idx]

                instrution = row["instrution"]
                question = row["question"]
                answer = row["answer"] if row["answer"] else self.no_answer_text

                positive_context = row["context"]
                positive_doc_id = int(self.context_to_doc_id[positive_context])

                ranked_doc_ids = indices[i]
                negative_docs = []

                for doc_id in ranked_doc_ids:
                    doc_id = int(doc_id)

                    if doc_id == positive_doc_id:
                        continue

                    doc_text = self.unique_contexts.iloc[doc_id]["context"]

                    # filter false negatives
                    if len(answer.split()) > 1 and answer.lower() in doc_text.lower():
                        continue

                    negative_docs.append(doc_text)
                    
                    if len(negative_docs) >= self.top_k:
                        break

                results.append({
                    "instrution": instrution,
                    "question": question,
                    "answer": answer,
                    "positive_docs": [positive_context],
                    "negative_docs": negative_docs,
                })

        return results
    
    @staticmethod
    def _mean_pooling(model_output, attention_mask):
        token_embeddings = model_output.last_hidden_state
        mask = attention_mask.unsqueeze(-1).float()
        summed = (token_embeddings * mask).sum(dim=1)
        counts = mask.sum(dim=1).clamp(min=1e-9)
        return summed / counts
    
    @torch.no_grad()
    def _embed_texts(self, texts: List[str]) -> np.ndarray:
        n = len(texts)
        d = self.model.config.hidden_size
        all_embeddings = np.empty((n, d), dtype=np.float32)
        idx = 0

        for start in tqdm(range(0, n, self.batch_size), desc='Embed_texts'):
            batch_texts = texts[start:start + self.batch_size]

            batch = self.tokenizer(
                batch_texts,
                max_length=self.max_length,
                truncation=True,
                padding=True,
                return_tensors="pt",
            ).to(self.device)

            outputs = self.model(**batch)

            embeddings = self._mean_pooling(outputs, batch["attention_mask"])
            embeddings = F.normalize(embeddings, p=2, dim=1)

            emb_np = embeddings.cpu().numpy()  # no astype needed if already float32

            bs = emb_np.shape[0]
            all_embeddings[idx:idx + bs] = emb_np
            idx += bs

        return all_embeddings

    def _embed_queries(self, texts: List[str]) -> np.ndarray:
        texts = [f"query: {text}" for text in texts]
        return self._embed_texts(texts)

    def _embed_contexts(self, texts: List[str]) -> np.ndarray:
        texts = [f"passage: {text}" for text in texts]
        return self._embed_texts(texts)

#### Build QA samples (qa_samples.json)

In [ ]:
import json

qa_builder = QASampleBuilder(
    df=full_df,
    model_name="intfloat/multilingual-e5-large",
    top_k=8,
    batch_size=4,
    max_length=512
)

qa_samples = qa_builder.build()

with open("qa_samples.json", "w", encoding="utf-8") as f:
    json.dump(qa_samples, f, ensure_ascii=False, indent=2)
    
print(qa_samples[0])

#### Class QADataset (create QA dataset from qa_samples.json)

In [41]:
import random
import torch
from torch.utils.data import Dataset


# template for getting special tokens length
TEMPLATE = """<BOS>
<INST></INST>

<CTX>
<D1></D>
<D2></D>
<D3></D>
<D4></D>
<D5></D>
<D6></D>
<D7></D>
<D8></D>
<D9></D>
</CTX>

<Q></Q>

<ANS></ANS>
<SRC></SRC>
<EOS>"""


class QADataset(Dataset):

    def __init__(
        self,
        data,
        tokenizer,
        max_seq_len=1024,
        max_answer=150
    ):
        self.data = data
        self.tokenizer = tokenizer
        self.tokenizer.post_processor = None
        self.max_seq_len = max_seq_len
        self.max_answer = max_answer
        self.teh_tokens_len = len(self.tokenizer.encode(TEMPLATE).ids)


    def _pack_documents(self, pos_doc, neg_docs, max_context_tokens):

        packed_docs = []
        total_tokens = 0
        
        pos_ids = self.tokenizer.encode(pos_doc).ids
        length = len(pos_ids)

        if length > max_context_tokens:
            packed_docs.append(self.tokenizer.decode(pos_ids[:max_context_tokens]))
            return packed_docs, 1

        packed_docs.append(pos_doc)
        total_tokens += length

        for doc in neg_docs:
            doc_ids = self.tokenizer.encode(doc).ids
            length = len(doc_ids)

            if total_tokens + length >= max_context_tokens:
                remain_tokens = max_context_tokens - total_tokens
                if remain_tokens > 10:
                    packed_docs.append(self.tokenizer.decode(doc_ids[:remain_tokens]))
                break

            packed_docs.append(doc)
            total_tokens += length

            if len(packed_docs) >= 9:
                break
 
        random.shuffle(packed_docs)
        pos_idx = packed_docs.index(pos_doc) + 1
        return packed_docs, pos_idx
    

    def build_prompt(self, docs, doc_idx, inst, question, answer):  
        context = ''
        for i, d in enumerate(docs):
            context += f"<D{i+1}>{d}</D>\n"

        prompt = (
            f"<BOS>\n"
            f"<INST>{inst}</INST>\n\n"
            f"<CTX>\n"
            f"{context}"
            f"</CTX>\n\n"
            f"<Q>{question}</Q>\n\n"
            f"<ANS>"
        )

        full_text = (
            f"{prompt}{answer}</ANS>\n"
            f"<SRC>{doc_idx}</SRC>\n"
            f"<EOS>"
        )

        return prompt, full_text
    

    def __len__(self):
        return len(self.data)
    

    def __getitem__(self, idx):

        sample = self.data[idx]

        inst = sample["instruction"] 
        inst_ids = self.tokenizer.encode(inst).ids

        question = sample["question"] 

        answer = sample["answer"]
        answer_ids = self.tokenizer.encode(answer).ids

        # --- truncate answer ---
        if len(answer_ids) > self.max_answer:
            answer_ids = answer_ids[:self.max_answer]
            answer = self.tokenizer.decode(answer_ids)

         # --- compute space left for context ---
        max_context_tokens = (
            self.max_seq_len
            - len(inst_ids)
            - len(answer_ids)
            - self.teh_tokens_len
            - len(self.tokenizer.encode(question).ids) - 3
        )

        # --- pack documents ---
        packed_docs, doc_idx = self._pack_documents(
            sample["positive_docs"][0], 
            sample["negative_docs"], 
            max_context_tokens
        )

        if len(packed_docs) == 1:
            if answer not in packed_docs[0]:
                answer = 'Ответ не найден.'

        prompt, full_text = self.build_prompt(packed_docs, doc_idx, inst, question, answer)

        prompt_ids = self.tokenizer.encode(prompt).ids
        full_ids = self.tokenizer.encode(full_text).ids
        full_ids_len = len(full_ids)

        if full_ids_len > self.max_seq_len:
            return None

        input_ids = torch.tensor(full_ids, dtype=torch.long)
        labels = input_ids.clone()

        # --- loss mask ---
        labels[:len(prompt_ids)] = -100

        return {
            "input_ids": input_ids,
            "labels": labels
        }
    

def qa_collate_fn(batch, pad_token_id):

    input_ids = [x["input_ids"] for x in batch]
    labels = [x["labels"] for x in batch]

    max_len = max(len(x) for x in input_ids)

    padded_inputs = []
    padded_labels = []

    for inp, lab in zip(input_ids, labels):

        pad_len = max_len - len(inp)

        padded_inputs.append(
            torch.cat([inp, torch.full((pad_len,), pad_token_id)])
        )

        padded_labels.append(
            torch.cat([lab, torch.full((pad_len,), -100)])
        )

    return {
        "input_ids": torch.stack(padded_inputs),
        "labels": torch.stack(padded_labels)
    }

#### Create QA Dataset and DataLoader

In [42]:
from torch.utils.data import DataLoader
import json

with open("qa_datasets/qa_dataset.json", "r", encoding="utf-8") as f:
    qa_dataset = json.load(f)

dataset = QADataset(
    data=qa_dataset,
    tokenizer=bpe_tokenizer,
    max_seq_len=1024
)

pad_id = bpe_tokenizer.token_to_id("<PAD>")

loader = DataLoader(
    dataset,
    batch_size=4,
    shuffle=True,
    collate_fn=lambda x: qa_collate_fn(x, pad_id)
)

In [17]:
from pathlib import Path
import sys

proj_root = Path("..").resolve()    # Path().resolve().parent
if str(proj_root) not in sys.path:
    sys.path.append(str(proj_root))
print(sys.path)

from tokenizer.bpe_tokenizer import BPETokenizer
bpe_tokenizer = BPETokenizer().from_file("d:/Neuro_Net/Projects/LLM/Gen/tokenizer/bpe_tokenizer_v2/tokenizer.json")

['c:\\Users\\Sergey\\AppData\\Local\\Programs\\Python\\Python312\\python312.zip', 'c:\\Users\\Sergey\\AppData\\Local\\Programs\\Python\\Python312\\DLLs', 'c:\\Users\\Sergey\\AppData\\Local\\Programs\\Python\\Python312\\Lib', 'c:\\Users\\Sergey\\AppData\\Local\\Programs\\Python\\Python312', '', 'C:\\Users\\Sergey\\AppData\\Roaming\\Python\\Python312\\site-packages', 'c:\\Users\\Sergey\\AppData\\Local\\Programs\\Python\\Python312\\Lib\\site-packages', 'D:\\Neuro_Net\\Projects\\LLM\\Gen']


In [19]:
print(bpe_tokenizer.get_vocab_size())
encoded = bpe_tokenizer.encode("Привет мир")
print(encoded.ids)
print(encoded.tokens)

32020
[22984, 309, 2245]
['ÐŁÑĢÐ¸Ð²', 'ÐµÑĤ', 'ĠÐ¼Ð¸ÑĢ']


In [43]:
dataset[1]

{'input_ids': tensor([    1,   201, 32000,  ..., 32019,   201,     2]),
 'labels': tensor([ -100,  -100,  -100,  ..., 32019,   201,     2])}

In [45]:
for i in range(len(dataset)):
    sample = dataset[i]
    ids = sample["input_ids"]
    real_len = (ids != 0).sum().item()

    if real_len < 600:
        print("Short sample:", i)
        print("Real tokens:", real_len)
        print("Total len:", len(ids))

        print("\nPrompt:")
        print(bpe_tokenizer.decode(ids.tolist()))
        break

Short sample: 6037
Real tokens: 586
Total len: 586

Prompt:

Ответь используя только данный контекст.


Королевские вооружённые силы Камбоджи — военная организация Камбоджи, предназначенная для защиты свободы, независимости и территориальной целостности государства. Состоят из королевских сухопутных войск, королевской жандармерии, королевских военно-морских и королевских военно-воздушных сил.
Вооружённые силы (ВС) — главная вооружённая организация государства или группы государств, предназначенная для обеспечения военной безопасности, защиты государственных интересов при агрессии и ведении войны, недопущения или ликвидации угрозы миру между государствами и безопасности. Кроме выполнения основных функций возложенных на вооружённые силы, они также могут привлекаться к поддержанию порядка в государстве при чрезвычайных ситуациях, ликвидации последствий природных и техногенных катастроф, а также для решения некоторых других государственных и международных задач.
Вооружённые силы Габона — в

In [20]:
batch = next(iter(loader))

print(batch["input_ids"].shape)
batch["input_ids"][1] 

torch.Size([8, 943])


tensor([    1,   201,    30,    37,    32,   201,    30,    38,    32, 24622,
         1149,  7661,  7494, 18272,  2687,   402, 24530,  1222, 14141,   402,
        25839,   538, 10932,  4818,  5910, 18272,   389, 18489,   287,  8805,
          280, 17635,    28, 30153,    14,  5910, 26927,  9611,    14, 12448,
          918,   287,  7455,    16, 15975,  2268,   402, 20063,  1222, 14141,
          402,  5487,   526, 14141,   261,   538,  5910,    14,   351, 31877,
          297, 17635,    14,  2838,   497,  8592,  4088,   725,  6740,    14,
          283,  5534,  2545, 18567,  7442,  5522,    16, 15975,  3763,  5222,
        27614,  4160, 10018,  9611,  4265,   262,   725,  3494,    16, 15975,
          274,  2685,   287, 10840, 21009,   538, 31067,   389,  4265,   345,
        29150,   283,  2146,  2411, 30153,   389,  8805,    16,    30,    17,
           38,    32,   201,    30,    38,    32,   592,   456,   918,  7923,
          538, 15782,   307,  7581,  5289,   399, 19904,  1023, 